In [1]:
import pandas as pd

df1 = pd.read_csv('/mnt/data/image_recognition/brown_forman_req/input/3k_pois.csv')

In [2]:
df1.columns
df1.rename(columns={'poi_code_primary': 'poi_code'}, inplace=True)

In [3]:
df2 = pd.read_csv('/mnt/data/image_recognition/brown_forman_req/output/restaurant_brand_data_include_all_attributes_mentions.csv')
df2.columns

Index(['poi_code', 'offerings__has_dancing',
       'service_options__has_seating_outdoors',
       'highlights__has_seating_rooftop', 'parking__has_parking_valet',
       'planning__recommends_reservations_lunch',
       'planning__recommends_reservations_brunch',
       'planning__recommends_reservations_dinner',
       'planning__requires_reservations', 'planning__accepts_reservations_x',
       'offerings__serves_happy_hour_food', 'atmosphere__feels_romantic',
       'offerings__has_private_dining_room', 'highlights__has_live_music',
       'offerings__serves_late_night_food',
       'offerings__serves_happy_hour_drinks_x', 'highlights__has_bar_games',
       'highlights__has_karaoke_nights', 'highlights__has_fast_service',
       'atmosphere__feels_upscale', 'atmosphere__feels_cozy',
       'atmosphere__feels_hip', 'atmosphere__is_recently_popular',
       'atmosphere__feels_casual', 'atmosphere__feels_quiet',
       'offerings__serves_wine', 'offerings__serves_liquor_x',
       '

In [4]:
df1.columns

Index(['poi_code', 'restaurant_name', 'location_status', 'reviews',
       'offerings__serves_cocktails', 'address', 'ratings', 'reviews_count',
       'offerings__has_dancing', 'service_options__has_seating_outdoors',
       'highlights__has_seating_rooftop', 'parking__has_parking_valet',
       'planning__recommends_reservations_lunch',
       'planning__recommends_reservations_brunch',
       'planning__recommends_reservations_dinner',
       'planning__requires_reservations', 'offerings__serves_happy_hour_food',
       'atmosphere__feels_romantic', 'offerings__has_private_dining_room',
       'highlights__has_live_music', 'offerings__serves_late_night_food',
       'highlights__has_bar_games', 'highlights__has_karaoke_nights',
       'highlights__has_fast_service', 'atmosphere__feels_upscale',
       'atmosphere__feels_cozy', 'atmosphere__feels_hip',
       'atmosphere__is_recently_popular', 'atmosphere__feels_casual',
       'atmosphere__feels_quiet', 'offerings__serves_wine',
   

In [5]:
# Standardize the restaurant name column before merging
# so both datasets contribute to a single restaurant_name field.
df1 = df1.rename(columns={'name': 'restaurant_name'}).copy()

# Treat blank strings as missing values so combine_first can fill from either dataset.
df1 = df1.replace(r'^\s*$', pd.NA, regex=True)
df2 = df2.replace(r'^\s*$', pd.NA, regex=True)


def coalesce_suffix_columns(df):
    df = df.copy()

    for col in list(df.columns):
        if col.endswith('_x'):
            base_col = col[:-2]
            y_col = f'{base_col}_y'

            if y_col in df.columns:
                df[base_col] = df[col].combine_first(df[y_col])
                df = df.drop(columns=[col, y_col])
            else:
                df = df.rename(columns={col: base_col})
        elif col.endswith('_y') and f"{col[:-2]}_x" not in df.columns:
            df = df.rename(columns={col: col[:-2]})

    return df


# Remove any existing _x/_y duplicates inside df2 first.
df2 = coalesce_suffix_columns(df2)

common_cols = [col for col in df1.columns if col in df2.columns and col != 'poi_code']

merged_df = df1.merge(df2, on='poi_code', how='outer', suffixes=('_df1', '_df2'))

# For columns present in both dataframes, keep one final column and
# fill missing values from whichever side has data.
for col in common_cols:
    merged_df[col] = merged_df[f'{col}_df1'].combine_first(merged_df[f'{col}_df2'])
    merged_df = merged_df.drop(columns=[f'{col}_df1', f'{col}_df2'])

# Keep poi_code and restaurant_name near the front for easier review.
front_cols = [col for col in ['poi_code', 'restaurant_name'] if col in merged_df.columns]
other_cols = [col for col in merged_df.columns if col not in front_cols]
merged_df = merged_df[front_cols + other_cols]

merged_df.head()

,poi_code,restaurant_name,filename,cost_for_two,absinthe_brand_name,absinthe_item_price,beer_brand_name,beer_item_price,brandy_brand_name,brandy_item_price,...,atmosphere__feels_cozy,atmosphere__feels_hip,atmosphere__is_recently_popular,atmosphere__feels_casual,atmosphere__feels_quiet,offerings__serves_wine,amenities__has_bar_onsite,planning__accepts_reservations,offerings__serves_happy_hour_drinks,offerings__serves_liquor
0,0x115aedf5dd67b353:0x7610f3ee573e0cda,Persian Darbar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,True,True,False,False,True,False,False
1,0x3a35e54988ce8c69:0x9c0d1b573a855ba0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,False,False
2,0x3bae16773e7dc413:0x46098eee49dc03f2,Symphony restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,True,False,True,True,True,True,True
3,0x3bc2b9ee2d315a49:0x780fe1c1dd51e23,Blue Tokai Coffee Roasters | Phoenix Marketcit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,False,True,False,False,False,False,False,False
4,0x3bddeba3a7e758cb:0x7ec0d6a656481ba8,mumbai_le-bar-salon-the-lounge-bar-mira-road,mumbai_le_bar_salon_the_lounge_bar_mira_road.json,"₹1,800 for two",NaN,NaN,other,399,other,749.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print(merged_df.shape)

filtered_df = merged_df[
    ~(
        merged_df['filename'].isna() &
        (merged_df['offerings__serves_liquor'] == False)
    )
]
print(filtered_df.shape)

(4525, 74)
(2402, 74)


In [11]:


filtered_df.to_csv('/mnt/data/image_recognition/brown_forman_req/output/restaurant_brand_data_include_all_attributes_mentions_2.csv', index=False)


In [8]:
df2['restaurant_name'].isna().sum()

np.int64(384)

In [ ]:
merged_df